# Project 1: Discharge Diagnosis Analysis

This notebook extracts discharge diagnoses from free-text ICU patient notes and summarizes them. Each of the 145 text files contains one patient's notes: a **discharge summary** (usually), radiology reports, and nursing progress notes. The goal is to:

1. Find the "Discharge Diagnosis" section in each discharge summary
2. Split it into individual diagnoses
3. Group synonyms and abbreviations (e.g. "afib" and "atrial fibrillation") under one standard name
4. Report the most common diagnoses, the number of patients with diagnoses, and the average number of diagnoses per patient

Because the notes are written by many different physicians, there is no single consistent format. Most of the work below is handling that variation.

## Step 1: Load the notes

Reads every `.txt` file in the `DischargeNotes` folder into a dictionary, keyed by file name (e.g. `NOTEEVENTS-19002`), with each patient's full notes as one long string. The summary statistics show how long the files are, which is a quick check that everything loaded correctly.

**Result:** 145 files loaded. File length varies widely, from 3,350 to 671,062 characters (median about 47,400). Longer files generally reflect longer ICU stays with more nursing notes.

In [1]:
from pathlib import Path
import re
from collections import Counter
import pandas as pd

DATA_DIR = Path("/Users/nityashankar/Desktop/bime300/DischargeNotes")
files = sorted(DATA_DIR.glob("*.txt"))
print(f"Found {len(files)} files")

notes = {f.stem: f.read_text(encoding="utf-8", errors="replace") for f in files}
pd.Series({k: len(v) for k, v in notes.items()}).describe()

Found 145 files


count       145.000000
mean      69981.834483
std       81104.663307
min        3350.000000
25%       20558.000000
50%       47402.000000
75%       88696.000000
max      671062.000000
dtype: float64

## Step 2: Explore the note structure

Before extracting anything, we need to know how the notes are organized. This cell finds every line that looks like a section header (text followed by a colon, like "Discharge Condition:") and counts how often each one appears across all files. The first list filters for headers related to discharge or diagnosis; the second shows the 60 most common headers overall.

This tells me (1) the different ways physicians label the diagnosis section and (2) which headers come right after it, so I know where to stop reading.

**Result:** The diagnosis section is labeled "discharge diagnosis" (69), "discharge diagnoses" (29), or occasionally "final diagnosis" (2). It is usually followed by "discharge condition" (67) or "discharge disposition" (64). "Admitting diagnosis" (1,848) comes from radiology report headers and is not a discharge diagnosis, so the extraction must specifically require the word "discharge."

In [2]:
header_re = re.compile(r"^\s*([A-Za-z][A-Za-z /&-]{2,40}):", re.MULTILINE)
header_counts = Counter()
for text in notes.values():
    header_counts.update(h.strip().lower() for h in header_re.findall(text))

for header, n in header_counts.most_common():
    if "discharg" in header or "diagnos" in header or "dx" in header:
        print(f"{n:4d}  {header}")

for header, n in header_counts.most_common(60):
    print(f"{n:4d}  {header}")

1848  admitting diagnosis
  83  discharge medications
  69  discharge diagnosis
  67  discharge condition
  64  discharge disposition
  63  discharge instructions
  29  discharge diagnoses
  26  discharge status
  22  condition on discharge
  14  medications on discharge
  12  preoperative diagnosis
   9  condition at discharge
   4  discharge instructions/followup
   2  final diagnosis
   2  discharge follow-up
   2  diagnosis
   2  discharge date
   2  labs on discharge
   2  secondary diagnosis
   2  primary diagnosis
   1  physical examination on discharge
   1  discharge followup
   1  discharge instructions-followup
   1  discharge disposition/condition
   1  admission diagnoses
   1  primary discharge diagnoses
   1  secondary discharge diagnoses
   1  discharge follow-up plans
   1  discharge physical examination
   1  initial diagnostic studies
   1  discharge activity
   1  discharge diet
   1  discharge services
   1  primary dx
   1  secondary dx
   1  nursing discharge not

## Step 2b: Count files with summaries and diagnosis sections

A rough count of how many files mention a discharge summary and how many contain a discharge diagnosis header anywhere in the text.

**Result:** Only 7 files contain the phrase "discharge summary," because most summaries are not labeled that way. They begin with "Admission Date:" instead, which appears 140 times, so roughly 140 patients have a summary. 99 files contain a discharge diagnosis header. The gap between these numbers is expected, since patients who died in the hospital have no discharge diagnoses.

In [3]:
dx_re = re.compile(r"discharge\s+diagnos[ie]s?", re.IGNORECASE)
summary_re = re.compile(r"discharge\s+summary", re.IGNORECASE)

print("Files mentioning a discharge summary:", sum(bool(summary_re.search(t)) for t in notes.values()))
print("Files with a discharge diagnosis header:", sum(bool(dx_re.search(t)) for t in notes.values()))

Files mentioning a discharge summary: 7
Files with a discharge diagnosis header: 99


## Step 3: Extract the discharge diagnosis section

For each file, this finds every "Discharge Diagnosis" (or "Final Diagnosis") header at the start of a line and captures the text after it. It stops at whichever comes first:

- A known following section (e.g. "Discharge Condition," "Discharge Instructions")
- Any other section header, except labels that appear *inside* diagnosis lists ("Primary Dx:", "Secondary:", "PMH:")
- A 1,500 character cap, as a backstop

Requiring the header at the start of a line avoids false matches from sentences like "see discharge diagnosis below." Capturing every match (not just the first) handles patients with an addendum or a second summary.

**Result:** 100 patients have a diagnosis section. Section length ranges from 4 to 708 characters (median 142), and none run into other parts of the note.

In [4]:
DX_START = re.compile(
    r"^\s*(?:discharge\s+diagnos[ie]s?|final\s+diagnos[ie]s?)\s*:?",
    re.IGNORECASE | re.MULTILINE,
)

STOP_HEADERS = [
    r"discharge condition", r"condition (?:on|at) discharge", r"discharge status",
    r"discharge disposition", r"discharge instructions", r"discharge follow\s*-?\s*up",
    r"discharge medications", r"medications on discharge", r"labs on discharge",
    r"physical examination on discharge", r"follow\s*-?\s*up", r"dictated by",
    r"completed by",
]
stop_re = re.compile(r"^\s*(?:" + "|".join(STOP_HEADERS) + r")\b",
                     re.IGNORECASE | re.MULTILINE)

GENERIC_HEADER_RE = re.compile(r"^\s*[A-Za-z][A-Za-z /&-]{2,40}:", re.MULTILINE)
SUBLABEL_OK = re.compile(
    r"^\s*(?:primary|secondary|other|pmh|past medical history)(?:\s+(?:diagnos[ie]s|dx))?\s*:",
    re.IGNORECASE,
)

def extract_dx_sections(text):
    sections = []
    for m in DX_START.finditer(text):
        rest = text[m.end():m.end() + 1500]  # hard cap
        cut = len(rest)

        stop = stop_re.search(rest)
        if stop:
            cut = min(cut, stop.start())

        for h in GENERIC_HEADER_RE.finditer(rest):
            line = rest[h.start():h.end()]
            if not SUBLABEL_OK.match(line):
                cut = min(cut, h.start())
                break

        section = rest[:cut].strip()
        if section:
            sections.append(section)
    return sections

dx_sections = {pid: extract_dx_sections(t) for pid, t in notes.items()}
dx_sections = {pid: s for pid, s in dx_sections.items() if s}

print(f"{len(dx_sections)} patients with at least one diagnosis section")
lengths = pd.Series({pid: sum(len(x) for x in s) for pid, s in dx_sections.items()})
print(lengths.describe())

100 patients with at least one diagnosis section
count    100.000000
mean     173.440000
std      135.059008
min        4.000000
25%       65.500000
50%      142.000000
75%      249.250000
max      708.000000
dtype: float64


## Step 4: Split each section into individual diagnoses

Physicians format diagnosis lists in several different ways, and this function handles each one:

- **Numbered lists** ("1. Aortic stenosis", "1.) Hypertension", "2- Atrial fibrillation"
- **One item per line with periods**, where long items wrap onto the next line ("Community acquired and hospital\nacquired pneumonia."). Lines are joined until a period ends the item.
- **One item per line with no punctuation** ("CAD\nPVD\nHTN"). Each line is its own item.
- **Comma-separated lists**, sometimes wrapping across lines ("Secondary Dx: C. diff colitis, Hypoxia-Induced\nSeizures, Anemia"). Items with two or more commas are split on commas. Items with only one comma are kept whole, since "CHF, systolic" is a single diagnosis.

It also removes the physician's signature block, de-identification brackets (`[**...**]`), sub-labels like "Primary Dx:", list numbers, and notes like "(Discharge diagnoses included)." Everything is lowercased so "Hypertension" and "hypertension" match.

In [5]:
DEID_RE = re.compile(r"\[\*\*.*?\*\*\]")
SIGNATURE_RE = re.compile(r"\bm\.?d\.?\b|\bdr\.|\bd:\s|\bt:\s|job\s*#", re.IGNORECASE)
LIST_MARK_RE = re.compile(r"^\s*(?:\d{1,2}[.)\-]+|[-*•])\s*")
SUBLABEL_RE = re.compile(
    r"^\s*(?:primary|secondary|other|pmh|past medical history)(?:\s+(?:diagnos[ie]s|dx))?\s*:\s*",
    re.IGNORECASE,
)

def split_dx(section):
    # 1. Cut off the signature and anything after it
    kept = []
    for line in section.splitlines():
        if SIGNATURE_RE.search(line) or line.strip().startswith("[**"):
            break
        kept.append(line.strip())

    # 2. Decide if this block ends items with periods or not
    nonempty = [l for l in kept if l]
    if not nonempty:
        return []
    uses_periods = sum(l.endswith(".") for l in nonempty) >= len(nonempty) / 2

    # 3. Group lines into items
    items, current = [], ""
    for line in kept:
        if not line:
            if current:
                items.append(current)
                current = ""
            continue
        in_comma_list = "," in current and not current.endswith(".")
        starts_new = (
            not current
            or LIST_MARK_RE.match(line)
            or SUBLABEL_RE.match(line)
            or current.endswith(".")
            or (not uses_periods and not in_comma_list)
        )
        if starts_new:
            if current:
                items.append(current)
            current = line
        else:
            current += " " + line
    if current:
        items.append(current)

    # 4. Split items crammed onto one line
    split_items = []
    for item in items:
        item = SUBLABEL_RE.sub("", item)
        for piece in re.split(r"\s(?=\d{1,2}[.)\-]+\s)|(?<=\.)\s+(?=[A-Za-z])", item):
            if piece.count(",") >= 2:
                split_items.extend(piece.split(","))
            else:
                split_items.append(piece)

    # 5. Clean each piece
    cleaned = []
    for piece in split_items:
        piece = piece.strip()
        if piece.startswith("(") and piece.endswith(")"):
            continue
        p = DEID_RE.sub(" ", piece)
        p = SUBLABEL_RE.sub("", p)
        p = LIST_MARK_RE.sub("", p)
        p = re.sub(r"\s+", " ", p).strip(" .,;:").lower()
        if "diagnos" in p:
            continue
        if sum(c.isalpha() for c in p) >= 2:
            cleaned.append(p)
    return cleaned

patient_dx = {
    pid: [dx for sec in secs for dx in split_dx(sec)]
    for pid, secs in dx_sections.items()
}

## Step 4b: Check the split on example patients

These six patients each represent a different list format, so they serve as test cases: 19002 (numbered list with a trailing signature), 19006 (items wrapping across lines), 19019 (abbreviations only), 19291 (starts with "PMH:"), 19380 (section that originally ran into the next part of the note), and 19494 (comma list with "Primary Dx" and "Secondary Dx" labels). The summary statistics flag any patient with 0 or 20+ items, which would suggest a section was cut too short or ran too long.

**Result:** All six split correctly. Patients have between 1 and 14 diagnoses (mean 5.52 before synonym grouping), and no patient has 0 or 20+ items.

In [6]:
for pid in ["NOTEEVENTS-19002", "NOTEEVENTS-19006", "NOTEEVENTS-19019", "NOTEEVENTS-19380"]:
    print(pid, patient_dx[pid], "\n")

counts = pd.Series({pid: len(d) for pid, d in patient_dx.items()})
print(counts.describe())
print("\nPatients with 0 items:", list(counts[counts == 0].index))
print("Patients with 20+ items:", list(counts[counts >= 20].index))

NOTEEVENTS-19002 ['aortic stenosis', 'coronary artery disease', 'insulin-dependent diabetes mellitus', 'hypertension', 'obesity', 'parkinson disease', 'depression', 'intraoperative atrial fibrillation'] 

NOTEEVENTS-19006 ['community acquired and hospital acquired pneumonia', 'hypoxia', 'acute respiratory distress syndrome', 'sepsis', 'hypotension (pressor dependent)', 'atrial fibrillation', 'gastroesophageal reflux disease', 'acute renal failure/chronic renal insufficiency', 'peripheral vascular disease', 'anemia', 'critical aortic stenosis', 'coronary artery disease'] 

NOTEEVENTS-19019 ['as', 'cad', 'pvd', 'htn', 'hyperlipidemia'] 

NOTEEVENTS-19380 ['myocardial infarction', 'aspiration pneumonia', 'small-bowel obstruction', 'dysphagia'] 

count    100.000000
mean       5.520000
std        3.421501
min        1.000000
25%        3.000000
50%        5.000000
75%        8.000000
max       14.000000
dtype: float64

Patients with 0 items: []
Patients with 20+ items: []


## Step 5: Define synonym rules

Each rule maps a set of patterns to one standard name. For example, "atrial fibrillation," "a fib," "afib," and "paf" all become "atrial fibrillation." Patterns include common abbreviations (CHF, CAD, HTN, DM, COPD), alternate wordings ("acute renal failure" and "acute kidney injury"), and misspellings in the notes.

There are 39 synonym groups in total.

In [7]:
SYNONYMS = [
    ("atrial fibrillation", r"atrial fibrillation|\ba\.?\s?fib\b|\bafib\b|\bpaf\b"),
    ("atrial flutter", r"atrial flutter|\ba\.?\s?flutter\b"),
    ("ventricular tachycardia", r"ventri\w* tachycardia|\bv\.?\s?tach\b|\bvt\b"),
    ("congestive heart failure", r"congestive heart failure|heart failure|\bchf\b"),
    ("coronary artery disease", r"coronary artery disease|coronary disease|\bcad\b"),
    ("myocardial infarction", r"myocardial infarction|\bn?stemi\b|\bmi\b"),
    ("aortic stenosis", r"aortic stenosis"),
    ("pulmonary hypertension", r"pulmonary hypertension|pulm(?:onary)? htn"),
    ("hypertension", r"(?<!pulmonary )hypertension|(?<!pulm )(?<!pulmonary )\bhtn\b"),
    ("hypotension", r"hypotension"),
    ("hyperlipidemia", r"hyperlipidemia|hypercholesterolemia|dyslipidemia|\bhld\b"),
    ("diabetes mellitus", r"diabetes mellitus|diabetes(?! insipidus)|\bn?iddm\b|\bt?2?dm2?\b"),
    ("peripheral vascular disease", r"peripheral vascular|peripheral arterial|\bpvd\b|\bpad\b"),
    ("copd", r"chronic obstructive pulmonary|\bcopd\b|emphysema"),
    ("pneumonia", r"pneumonia|\bpna\b"),
    ("acute respiratory distress syndrome", r"acute respiratory distress|\bards\b"),
    ("respiratory failure", r"respiratory failure"),
    ("pulmonary edema", r"pulmonary edema"),
    ("obstructive sleep apnea", r"obstructive (?:sleep )?apnea|sleep apnea|\bosa\b"),
    ("sepsis", r"sepsis|septic"),
    ("seizures", r"seizure"),
    ("hypoxia", r"\bhypox(?:ia|emia)\b"),
    ("acute kidney injury", r"acute renal failure|acute kidney injury|\barf\b|\baki\b"),
    ("chronic kidney disease", r"chronic renal|chronic kidney|\bckd\b|\bcri\b|\besrd\b|end.stage renal"),
    ("urinary tract infection", r"urinary tract infection|\buti\b"),
    ("gastrointestinal bleed", r"\bgi bleed|gastrointestinal (?:bleed|hemorrhage)|\b[ul]gib\b"),
    ("gerd", r"gastroesophageal reflux|\bgerd\b"),
    ("clostridium difficile infection", r"clostridium|c\.?\s?diff"),
    ("deep vein thrombosis", r"deep ve(?:in|nous) thrombosis|\bdvt\b"),
    ("pulmonary embolism", r"pulmonary embol|\bpe\b"),
    ("stroke", r"stroke|cerebr(?:o|al) ?vascular accident|\bcva\b"),
    ("transient ischemic attack", r"transient ischemi\w* attack|\btia\b"),
    ("osteoarthritis", r"osteoarthritis|degenerative joint disease|\bdjd\b"),
    ("pressure ulcer", r"decubitus|pressure ulcer"),
    ("dementia", r"dementia"),
    ("hypothyroidism", r"hypothyroid"),
    ("gout", r"\bgout\b"),
    ("anemia", r"anemia"),
    ("depression", r"depression"),
]
SYNONYMS = [(name, re.compile(pat)) for name, pat in SYNONYMS]

EXACT_ONLY = {"as": "aortic stenosis", "af": "atrial fibrillation"}

def expand(dx):
    """Remove parentheticals and split 'x/y' combos into two items (but keep 's/p', 'h/o', 'w/')."""
    dx = re.sub(r"\(.*?\)", "", dx)
    dx = re.sub(r"\s+", " ", dx).strip(" .,;:-")
    parts = [p.strip() for p in re.split(r"(?<!\bs)(?<!\bh)(?<!\bw)/", dx)]
    if len(parts) > 1 and all(len(p) > 3 for p in parts):
        return parts
    return [dx]

def canonicalize(dx):
    """Return every standard name this diagnosis matches (empty list if none)."""
    if dx in EXACT_ONLY:
        return [EXACT_ONLY[dx]]
    return [name for name, pattern in SYNONYMS if pattern.search(dx)]

## Step 6: Count diagnoses per patient and build the table

Each patient's diagnoses are converted to standard names and stored as a set, so each diagnosis counts **once per patient** even if it appears multiple times (e.g. both "CAD" and "coronary artery disease"). Diagnoses that match no rule keep their cleaned text as their name.

**Result:**
- **100 patients** have discharge diagnoses
- **5.71 diagnoses per patient** on average (among patients with at least one)
- Most common: hypertension (35 patients), coronary artery disease (24), atrial fibrillation, diabetes mellitus, and congestive heart failure (19 each), anemia (16), pneumonia and myocardial infarction (13 each), acute kidney injury (11), and dementia (10)

This is consistent with a typical ICU population, where cardiovascular and chronic conditions dominate.

In [8]:
patient_canon = {}
unmatched = Counter()

for pid, dxs in patient_dx.items():
    names = set()
    for dx in dxs:
        for piece in expand(dx):
            found = canonicalize(piece)
            if found:
                names.update(found)
            else:
                names.add(piece)
                unmatched[piece] += 1
    patient_canon[pid] = names

dx_counts = Counter(name for names in patient_canon.values() for name in names)
table = pd.DataFrame(dx_counts.most_common(), columns=["Discharge diagnosis", "Patients"])

n_patients = len(patient_canon)
avg_dx = sum(len(n) for n in patient_canon.values()) / n_patients

print(f"Patients with discharge diagnoses: {n_patients}")
print(f"Average diagnoses per patient: {avg_dx:.2f}")
table.head(25)

Patients with discharge diagnoses: 100
Average diagnoses per patient: 5.71


,Discharge diagnosis,Patients
0,hypertension,35
1,coronary artery disease,24
2,diabetes mellitus,19
3,atrial fibrillation,19
4,congestive heart failure,19
5,anemia,16
6,pneumonia,13
7,myocardial infarction,13
8,acute kidney injury,11
9,dementia,10


## Step 7: Review diagnoses with no synonym rule

Lists diagnoses that did not match any rule, most frequent first. I used this list to find synonyms I had missed: earlier versions showed "pulmonary edema" and "flash pulmonary edema" counted separately, "osteoarthritis" and "degenerative joint disease" counted separately, and two spellings of "clostridium difficile." Each of those led to a new rule.

**Result:** The remaining unmatched diagnoses are mostly conditions that appear once or twice and have no common synonyms, such as dysphagia, constipation, and failure to wean off of ventilator.

In [9]:
for name, n in unmatched.most_common(40):
    print(f"{n:3d}  {name}")

  3  dysphagia
  2  s/p ablation
  2  constipation
  2  urinary incontinence
  2  esophagitis
  2  lung cancer
  2  respiratory distress
  2  hyperkalemia
  2  failure to thrive
  2  abdominal aortic aneurysm
  2  anxiety
  2  hypernatremia
  2  pancytopenia
  2  metastatic melanoma
  2  chest pain
  2  chronic pain
  1  obesity
  1  parkinson disease
  1  fevers of unknown source
  1  failure to wean off of ventilator
  1  fluid overload
  1  diverticulosis
  1  sjogrn's syndrome
  1  raynaud's
  1  s/p tunneled line
  1  severe hypoglycemia
  1  s/p recent metatarsal amputation
  1  x 3 (lima to lad
  1  svg to om
  1  diag)
  1  maze procedure
  1  ligation
  1  tricuspid valve repair w/ 28mm ce annuloplasty ring
  1  hepatic encephalopathy
  1  status post endoscopy with banding of esophageal varices
  1  status post abdominal paracentesis
  1  ascites
  1  hepatorenal syndrome
  1  nonsmall cell lung cancer, metastatic to the brain
  1  status post pericardial tamponade, constrict

## Step 8: Check what each synonym rule matched

Step 7 shows what the rules missed; this cell shows what they caught. For each of the top 25 diagnoses, it lists every original phrase that was grouped under it. This is how I checked for false matches, where a pattern catches something it should not.

**Result:** No false matches found. The grouping is working as intended. For example:
- **Atrial fibrillation** combines 11 different phrasings, including "afib," "a fib," "paroxysmal atrial fibrillation," and "atrial fibrillation with rvr"
- **Diabetes mellitus** combines "dm," "niddm," "type 2 diabetes," "insulin dependent diabetes mellitus," and "dm ii with neuropathy"
- **Stroke** combines "cerebrovascular accident," "cerebral vascular accident," "hx cva," and "acute hemorrhagic cerebellar stroke"

In [10]:
from collections import defaultdict

mapped_from = defaultdict(Counter)
for dxs in patient_dx.values():
    for dx in dxs:
        for piece in expand(dx):
            for name in canonicalize(piece):
                if piece != name:
                    mapped_from[name][piece] += 1

for name in table["Discharge diagnosis"].head(25):
    if name in mapped_from:
        print(f"\n{name.upper()}")
        for raw, n in mapped_from[name].most_common():
            print(f"  {n:3d}  {raw}")


HYPERTENSION
    6  htn
    1  patch with pericardium to lv wall on hypertension hyperlipidemia transient ischemia attack
    1  hypotension, sirs dementia bph hypertension
    1  hypertension secondary to sutent

CORONARY ARTERY DISEASE
    3  cad
    1  severe two vessel coronary artery disease
    1  coronary artery disease/paf/tr s/p coronary artery bypass graft
    1  coronary artery disease with non-st elevation myocardial
    1  coronary artery disease s/p cabg
    1  coronary artery disease status post two myocardial infarctions
    1  coronary artery disease status post coronary artery bypass grafting with a left internal mammary artery to the left anterior descending artery, saphenous vein graft to posterior descending artery and saphenous vein graft to obtuse marginal
    1  coronary artery disease s/p coronary artery bypass graft x 2
    1  cad s/p cabg
    1  cad with ef 50%
    1  known coronary artery disease
    1  cad with echo evidence of prior mi and reduced ef
    

## Step 9: Validate patients without diagnoses

Of the 145 files, 45 have no extracted diagnoses. This cell checks why: whether they have a discharge summary at all, and if so, whether the notes mention the patient dying (in which case no discharge diagnoses are expected).

**Result:** Of the 45 patients without diagnoses, 40 have a discharge summary, and 8 of those mention the patient expiring. 5 have no discharge summary at all. Note that "expired" can also appear in nursing notes, so this count is approximate.

In [11]:
no_dx = [pid for pid in notes if pid not in patient_canon]
has_summary = [pid for pid in no_dx
               if re.search(r"^\s*admission date", notes[pid], re.IGNORECASE | re.MULTILINE)]
expired = [pid for pid in has_summary
           if re.search(r"\bexpired\b|\bdeceased\b|passed away", notes[pid], re.IGNORECASE)]

print(f"Total files: {len(notes)}")
print(f"With discharge diagnoses: {len(patient_canon)}")
print(f"Without diagnoses: {len(no_dx)}")
print(f"  ...that have a discharge summary: {len(has_summary)}")
print(f"  ...of those, mention the patient expiring: {len(expired)}")
print(f"  ...with no discharge summary at all: {len(no_dx) - len(has_summary)}")

Total files: 145
With discharge diagnoses: 100
Without diagnoses: 45
  ...that have a discharge summary: 40
  ...of those, mention the patient expiring: 8
  ...with no discharge summary at all: 5


## Summary and limitations

**Final results:** 100 of 145 patients have discharge diagnoses, with an average of 5.71 diagnoses each. The most common are hypertension (35), coronary artery disease (24), and atrial fibrillation, diabetes mellitus, and congestive heart failure (19 each).

**To note:**
- History-of items the physician listed under discharge diagnoses ("h/o upper GI bleed," "s/p CABG") are counted, since the physician chose to include them.
- Subtypes are grouped under the general condition: all pneumonia types together; septic shock and urosepsis under sepsis; afib with rapid ventricular response under atrial fibrillation.
- An item naming more than one condition counts toward each ("CAD status post two MIs" counts as both coronary artery disease and myocardial infarction).

**Limitations:**
- Synonym matching depends on rules I wrote by reviewing this dataset, so rare abbreviations or misspellings may still be missed.
- A few groupings are debatable. "Hypoxia-induced seizures" counts as both hypoxia and seizures, and "chronic anemia of heart failure" counts as both anemia and heart failure.
- Run-on items with no separators may still lose conditions that have no synonym rule (e.g. "bph" in "hypotension, sirs dementia bph hypertension").
- A more robust approach would map diagnoses to a standard terminology like ICD-10 or UMLS using a clinical NLP tool, rather than hand-written rules.